In [ ]:
import pandas as pd
import pickle
from tqdm import tqdm
import re
from wordcloud import WordCloud, STOPWORDS

**Note:** This notebook was originally developed in Google Colab. It now runs locally; install the sentiment extra first (`uv sync --extra sentiment --extra viz`). A GPU is used automatically when available.

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
#import spacy

In [ ]:
import torch
from transformers import pipeline
from datasets import Dataset

In [ ]:
classifier = pipeline("sentiment-analysis",model='neuraly/bert-base-italian-cased-sentiment',truncation=True,
    max_length=512,device=0 if torch.cuda.is_available() else -1)

In [ ]:
classifier.batch_size = 100  # increase the batch size to reduce the number of batches
classifier.model.config.max_length = 512  # increase the sequence length to process longer texts

In [ ]:
from pathlib import Path

from subreddit_lens import load_comments, preprocess

# Locate the repository root so paths work regardless of the working directory.
REPO_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "output"
filename = DATA_DIR / "litigi_comments.parquet"

In [ ]:
litigi = load_comments(filename)

In [ ]:
tqdm.pandas()

In [ ]:
#nlp = spacy.load('it_core_news_lg')

In [ ]:
#doc = nlp("	Number one: perle è uno state of mind, non un subreddit. Non ce ne frega un cazzo che non sei in grado di seguire le regole di un subreddit, per quanto stupide. Fondati il tuo e vedi. Ah, già: lo hai fatto e non ti si incula nessuno. Number two: la lista dei moderatori è visibile nella front page di ogni subreddit. Usi roba tipo script, python e cazzi pitonati e non sai come recuperare queste informazioni base? Chi vuoi prendere per il culo? La verità (come ho detto già in passato) è che sei un drogato di attenzione, faresti di tutto per avere mezza interazione. Il problema è che sei tremendamente banale e noioso, la cosa più stimolante che riesci a fare è spammare un brusio continuo di merda insignificante finche qualcuno (me in questo caso) non smatta e ti risponde più per sfinimento che per pietà. Vaffanculo Pietro, se avessi un minimo di saggezza, lavoreresti su te stesso. Mi fai rimpiangere i mentecatti del covid, renditi conto. Sei solo chiacchiera indistinta, l'incubo di Heidegger.")
#displacy.render(doc, style="ent",jupyter=True)


In [ ]:
litigi['body_preprocessed']=litigi['body'].apply(lambda x: preprocess(x))


In [ ]:
classifier("Number one: perle è uno state of mind, non un subreddit. Non ce ne frega un cazzo che non sei in grado di seguire le regole di un subreddit, per quanto stupide. Fondati il tuo e vedi. Ah, già: lo hai fatto e non ti si incula nessuno. Number two: la lista dei moderatori è visibile nella front page di ogni subreddit. Usi roba tipo script, python e cazzi pitonati e non sai come recuperare queste informazioni base? Chi vuoi prendere per il culo? La verità (come ho detto già in passato) è che sei un drogato di attenzione, faresti di tutto per avere mezza interazione. Il problema è che sei tremendamente banale e noioso, la cosa più stimolante che riesci a fare è spammare un brusio continuo di merda insignificante finche qualcuno (me in questo caso) non smatta e ti risponde più per sfinimento che per pietà. Vaffanculo Pietro, se avessi un minimo di saggezza, lavoreresti su te stesso. Mi fai rimpiangere i mentecatti del covid, renditi conto. Sei solo chiacchiera indistinta, l'incubo di Heidegger.")

In [ ]:
litigi[['body_preprocessed','body','author']]

In [ ]:
#for word in doc:
#    lexeme = doc.vocab[word.text]
#    print(lexeme.text, lexeme.orth, lexeme.shape_, lexeme.prefix_, lexeme.suffix_,
#            lexeme.is_alpha, lexeme.is_digit, lexeme.is_title, lexeme.lang_)

In [ ]:
litigi=litigi.reset_index(drop=True)

In [ ]:
print(litigi.iloc[[2487]]['body_preprocessed'])

In [ ]:
litigi['sentiment'] = litigi['body_preprocessed'].progress_apply(lambda x: classifier(x)[0]['label'])

In [ ]:
litigi[litigi['author']=='ohnoitsalreadytaken'][['author','body','sentiment']]


In [ ]:
#
def dataset_processing():
    '''creates a dataset from pandas df and processes it, returns a df'''
    dataset = Dataset.from_pandas(litigi[['body','author','created_utc','subreddit','id','permalink','link_id','body_preprocessed']])
    # apply the sentiment analysis pipeline to the dataset
    output = classifier(dataset['body_preprocessed'])
    # add the sentiment analysis output to the dataset
    dataset = dataset.add_column('sentiment', output)
    out_df=dataset.to_pandas()

In [ ]:
out_df=litigi

In [ ]:
out_df['sentiment_result']=out_df['sentiment'].apply(lambda x:x['label'])

In [ ]:
out_df['num_words'] = out_df['body_preprocessed'].str.split().apply(len)

In [ ]:
out_df[out_df['author']=='ohnoitsalreadytaken' ][['author','body','sentiment','num_words']]


In [ ]:
#myvals = out_df['body_preprocessed'].values.tolist()

#out_df['number_of_words_spacy'] = [len([i for i in nlp(re.sub('</b>|<b>', '', i2)) if not (i.is_space or i.is_punct)]) for i2 in myvals]

In [ ]:
out_df[['sentiment','sentiment_result']]

In [ ]:
df_pivot.sort_values('positive').to_csv(OUTPUT_DIR / 'user_sentiment_litigi.csv')

In [ ]:
out_df.to_pickle(OUTPUT_DIR / 'out_sentiment.pickle')

In [ ]:
# out_df.to_pickle(OUTPUT_DIR / 'out_sentiment.pickle')  # already saved above in cell-35

In [ ]:
#litigi['sentiment'] = litigi['body_preprocessed'].progress_apply(lambda x: classifier(x)[0]['label'])